In [1]:
!pip install ddgs sentence-transformers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 100.0 MB/s eta 0:00:0000:01


In [2]:
import re
import torch
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

In [8]:
class ClaimExtractor:

    def extract_claims(self, article):

        sentences = re.split(r'[.!?]+', article)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

        claims = sentences[:3]

        return claims

In [18]:
class QueryGenerator:

    def generate_queries(self, claims):

        queries = []

        for claim in claims:

            keywords = re.findall(r"[A-Za-z0-9-]+", claim)

            keywords = [
                w for w in keywords
                if len(w) > 2
            ]

            short = " ".join(keywords[:8])

            queries.append(short)

            queries.append(short + " Reuters")

            queries.append(short + " official")

        return queries

In [19]:
from ddgs import DDGS
from urllib.parse import urlparse

class KnowledgeRetriever:

    def __init__(self):

        # ✅ Expanded credible domains
        self.credible_domains = {

            # High trust news
            "reuters.com":1.0,
            "apnews.com":1.0,
            "bbc.com":1.0,
            "bbc.co.uk":1.0,

            # Major media
            "nytimes.com":0.95,
            "theguardian.com":0.95,
            "washingtonpost.com":0.95,
            "wsj.com":0.95,
            "bloomberg.com":0.95,

            # Fact-checking ⭐
            "snopes.com":1.0,
            "factcheck.org":1.0,
            "politifact.com":1.0,

            # Science / health
            "nature.com":0.95,
            "science.org":0.95,
            "sciencedaily.com":0.9,
            "nih.gov":1.0,
            "who.int":1.0,
            "cdc.gov":1.0,

            "nasa.gov":1.0,
            "science.nasa.gov":1.0,
            "jpl.nasa.gov":1.0,
            "isro.gov.in":1.0,"esa.int":1.0,

            # Educational / government
            ".edu":0.9,
            ".gov":1.0
        }

    # ✅ Extract domain cleanly
    def extract_domain(self, url):

        try:
            domain = urlparse(url).netloc.lower()
            domain = domain.replace("www.", "")
            return domain
        except:
            return ""

    # ✅ Assign credibility score
    def get_credibility(self, domain):

        for d, score in self.credible_domains.items():
            if d in domain:
                return score

        return 0.3  # default low

    # ✅ Detect source type (optional but useful)
    def get_source_type(self, domain):

        if ".gov" in domain or ".edu" in domain:
            return "HIGH_TRUST"

        if any(d in domain for d in ["reuters","bbc","apnews"]):
            return "NEWS"

        if any(d in domain for d in ["snopes","factcheck","politifact"]):
            return "FACT_CHECK"

        return "UNKNOWN"

    # ✅ MAIN SEARCH FUNCTION
    def search(self, query):

        evidence = []

        if query.strip() == "":
            return evidence

        try:
            with DDGS() as ddgs:

                results = ddgs.text(query, max_results=20)

                for r in results:

                    url = r.get("href", "")
                    title = r.get("title", "")
                    snippet = r.get("body", "")

                    domain = self.extract_domain(url)

                    credibility = self.get_credibility(domain)

                    # ✅ FILTER LOW QUALITY
                    if credibility < 0.6:
                        continue

                    evidence.append({
                        "title": title,
                        "snippet": snippet,
                        "url": url,
                        "domain": domain,
                        "credibility_score": credibility,
                        "source_type": self.get_source_type(domain)
                    })

        except Exception as e:
            print("Search Error:", e)

        # ✅ REMOVE DUPLICATES
        unique = []
        seen = set()

        for e in evidence:
            if e["url"] not in seen:
                unique.append(e)
                seen.add(e["url"])

        return unique

In [26]:
class EvidenceRanker:

    def __init__(self):

        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if not evidence:
            return []

        claim_embedding = self.encoder.encode(claim, convert_to_tensor=True)

        snippets = [e["snippet"] for e in evidence]

        snippet_embeddings = self.encoder.encode(snippets, convert_to_tensor=True)

        similarities = util.cos_sim(claim_embedding, snippet_embeddings)[0]

        for i,e in enumerate(evidence):

            semantic_score = float(similarities[i])

            e["score"] = 0.7*semantic_score + 0.3*e["credibility_score"]

        ranked = sorted(evidence, key=lambda x:x["score"], reverse=True)

        return ranked[:5]

In [27]:
class ClaimVerifier:

    def __init__(self):

        model_name = "mistralai/Mistral-7B-Instruct-v0.2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    def verify(self, claim, evidence):

        # ---------- Handle empty evidence ----------
        if len(evidence) == 0:
            return ( "NOT ENOUGH EVIDENCE",0.25,"No reliable evidence found.")

        # ---------- Prepare evidence ----------
        top_evidence = evidence[:3]
        evidence_text = "\n".join([e["snippet"] for e in top_evidence])

        # ---------- Improved Prompt ----------
        prompt = f"""
You are a fact-checking assistant.

Claim:
{claim}

Evidence:
{evidence_text}

Task:
- Check whether evidence supports or contradicts the claim
- Use only given evidence

Respond EXACTLY like this:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation based on evidence
"""

        # ---------- Tokenize ----------
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # ---------- Generate ----------
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # ---------- Default values ----------
        verdict = "REAL"
        confidence = 0.5
        explanation = "Could not determine clearly."

        # ---------- Safe Parsing ----------
        for line in response.split("\n"):

            line_upper = line.upper()

            if "VERDICT" in line_upper:
                if "FALSE" in line_upper:
                    verdict = "FAKE"
                elif "TRUE" in line_upper:
                    verdict = "REAL"

            elif "CONFIDENCE" in line_upper:
                import re
                nums = re.findall(r"\d*\.?\d+", line)
                if nums:
                    confidence = float(nums[0])

            elif "EXPLANATION" in line_upper:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    explanation = parts[1].strip()

        # ---------- Add Source-Based Explanation ----------
        high_cred_sources = [e for e in evidence if e["credibility_score"] >= 0.9]
        domains = [e["domain"] for e in top_evidence if "domain" in e]

        if verdict == "REAL":
            explanation += f" Supported by {len(high_cred_sources)} high-credibility sources."
        else:
            explanation += f" Contradicted by credible sources."

        if domains:
            explanation += f" Sources include: {', '.join(domains[:2])}."

        return verdict, confidence, explanation

In [28]:
class OutputGenerator:

    def display(self, claim, verdict, confidence, explanation, evidence):

        print("\n==============================")
        print("FAKE NEWS DETECTION RESULT")
        print("==============================")

        print("\nClaim:")
        print(claim)

        print("\nFinal Verdict:", verdict)

        print("Confidence:", round(confidence,2))

        print("\nExplanation:")
        print(explanation)

        print("\nTop Evidence Sources:")

        for i,e in enumerate(evidence[:3],1):

            print(f"{i}. {e['title']}")
            print(f"   {e['url']}\n")

        print("==============================")

In [29]:
class FakeNewsPipeline:

    def __init__(self):

        self.extractor = ClaimExtractor()
        self.query_gen = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()
        self.output = OutputGenerator()

    # ---------- FINAL EXPLANATION FUNCTION ----------
    def generate_final_explanation(self, details, final_verdict):

        real_count = sum(1 for d in details if d["verdict"] == "REAL")
        fake_count = sum(1 for d in details if d["verdict"] == "FAKE")

        if final_verdict == "REAL":
            return "All major claims are supported by high-credibility sources."

        elif final_verdict == "FAKE":
            return "Most claims are contradicted by credible sources."

        else:
            return (
                "The article contains a mix of true and false information. "
                f"{real_count} claims are supported, while {fake_count} claims are not supported."
            )

    # ---------- MAIN PIPELINE ----------
    def detect(self, article):

        claims = self.extractor.extract_claims(article)

        if not claims:
            return {
                "verdict": "UNVERIFIED",
                "confidence": 0.0,
                "details": []
            }

        all_results = []
        all_evidence = []

        # ---------- PROCESS EACH CLAIM ----------
        for claim in claims:

            queries = self.query_gen.generate_queries([claim])

            evidence = []

            for q in queries:
                if q.strip() == "":
                    continue
                evidence.extend(self.retriever.search(q))

            ranked = self.ranker.rank(claim, evidence)

            verdict, confidence, explanation = self.verifier.verify(claim, ranked)

            all_results.append({
                "claim": claim,
                "verdict": verdict,
                "confidence": confidence,
                "explanation": explanation
            })

            all_evidence.extend(ranked[:2])

        # ---------- FINAL DECISION ----------
        fake_count = sum(1 for r in all_results if r["verdict"] == "FAKE")
        real_count = sum(1 for r in all_results if r["verdict"] == "REAL")

        if fake_count > 0 and real_count > 0:
            final_verdict = "PARTIALLY FAKE"
        elif fake_count > 0:
            final_verdict = "FAKE"
        else:
            final_verdict = "REAL"

        avg_conf = sum(r["confidence"] for r in all_results) / len(all_results)

        # ✅ Normalize confidence (important for research)
        avg_conf = min(avg_conf, 0.85)

        # ---------- FINAL EXPLANATION ----------
        final_explanation = self.generate_final_explanation(all_results, final_verdict)

        # ---------- DISPLAY ----------
        print("\n==============================")
        print("FINAL ARTICLE VERDICT")
        print("==============================")

        print("\nFinal Verdict:", final_verdict)
        print("Confidence:", round(avg_conf, 2))

        if final_verdict == "PARTIALLY FAKE":
            print("\n⚠️ WARNING: Article contains mixed or misleading information")

        # ✅ NEW ADDITION
        print("\nFinal Explanation:")
        print(final_explanation)

        print("\n--- CLAIM LEVEL ANALYSIS ---")

        for i, r in enumerate(all_results, 1):
            print(f"\n{i}. Claim: {r['claim']}")
            print(f"   Verdict: {r['verdict']}")
            print(f"   Confidence: {round(r['confidence'], 2)}")
            print(f"   Explanation: {r['explanation']}")

        print("\n--- TOP SOURCES ---")

        for i, e in enumerate(all_evidence[:5], 1):
            print(f"{i}. {e['title']}")
            print(f"   {e['url']}")

        print("==============================")

        # ---------- RETURN ----------
        return {
            "verdict": final_verdict,
            "confidence": avg_conf,
            "final_explanation": final_explanation,   # ✅ NEW
            "details": all_results
        }

In [23]:
pipeline = FakeNewsPipeline()

article = """
Scientists report that moderate coffee consumption (3–4 cups daily) may improve longevity.
However, some viral posts claim drinking 10 cups of coffee daily can extend lifespan by 50 years.
Health experts warn that excessive caffeine intake can cause heart problems and anxiety.
"""

result = pipeline.detect(article)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


FINAL ARTICLE VERDICT

Final Verdict: PARTIALLY FAKE
Confidence: 0.85

⚠️ WARNING: Article contains mixed or misleading information

Final Explanation:
The article contains a mix of true and false information. 2 claims are supported, while 1 claims are not supported.

--- CLAIM LEVEL ANALYSIS ---

1. Claim: Scientists report that moderate coffee consumption (3–4 cups daily) may improve longevity
   Verdict: REAL
   Confidence: 0.9
   Explanation: The evidence suggests that moderate coffee consumption (3-5 cups daily) is safe for most adults and may even have health benefits, including reducing the risk of cardiovascular disease. Supported by 5 high-credibility sources. Sources include: washingtonpost.com, pmc.ncbi.nlm.nih.gov.

2. Claim: However, some viral posts claim drinking 10 cups of coffee daily can extend lifespan by 50 years
   Verdict: FAKE
   Confidence: 1.0
   Explanation: The evidence provided does not mention anything about coffee consumption extending lifespan by 50 year

In [24]:
!pip install ipywidgets

In [30]:
import ipywidgets as widgets
from IPython.display import display

# Text area for input
input_box = widgets.Textarea(
    value="",
    placeholder="Enter news article here...",
    description="Article:",
    layout=widgets.Layout(width='100%', height='150px')
)

# Button
run_button = widgets.Button(
    description="Check Fake News",
    button_style='success'
)

# Output area
output_area = widgets.Output()

# Function on click
def on_button_click(b):
    with output_area:
        output_area.clear_output()

        article = input_box.value.strip()

        if article == "":
            print("⚠️ Please enter an article")
            return

        print("⏳ Processing...\n")

        result = pipeline.detect(article)   # your pipeline

# Attach event
run_button.on_click(on_button_click)

# Display UI
display(input_box, run_button, output_area)

Textarea(value='', description='Article:', layout=Layout(height='150px', width='100%'), placeholder='Enter new…

Button(button_style='success', description='Check Fake News', style=ButtonStyle())

Output()

In [32]:
import re
from urllib.parse import urlparse
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util

# =========================
# CLAIM EXTRACTOR
# =========================
class ClaimExtractor:
    def extract_claims(self, article):
        sentences = re.split(r'[.!?]+', article)
        return [s.strip() for s in sentences if len(s.strip()) > 25][:3]


# =========================
# QUERY GENERATOR
# =========================
class QueryGenerator:
    def generate_queries(self, claims):
        queries = []
        for c in claims:
            queries += [
                c,
                c + " fact check",
                "Is it true that " + c
            ]
        return queries[:5]


# =========================
# KNOWLEDGE RETRIEVER
# =========================
class KnowledgeRetriever:

    def extract_domain(self, url):
        return urlparse(url).netloc.replace("www.", "").lower()

    def clean_domain(self, d):
        if "bbc" in d: return "BBC"
        if "cnn" in d: return "CNN"
        if "reuters" in d: return "Reuters"
        if "pubmed" in d: return "PubMed"
        if "nih" in d: return "NIH"
        if "who" in d: return "WHO"
        if "cdc" in d: return "CDC"
        if "nature" in d: return "Nature"
        if "healthline" in d: return "Healthline"
        if "medicalnewstoday" in d: return "MedicalNewsToday"
        if "indiatimes" in d: return "Economic Times"
        return d.split(".")[0].capitalize()

    def search(self, query):
        evidence = []
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=8)

            for r in results:
                url = r.get("href", "")
                raw = self.extract_domain(url)

                evidence.append({
                    "snippet": r.get("body", ""),
                    "url": url,
                    "domain": self.clean_domain(raw)
                })

        # remove duplicates
        seen = set()
        unique = []
        for e in evidence:
            if e["url"] not in seen:
                unique.append(e)
                seen.add(e["url"])

        return unique[:6]


# =========================
# RANKER + SOURCE ATTRIBUTION CORE
# =========================
class EvidenceRanker:

    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if not evidence:
            return []

        claim_emb = self.model.encode(claim, convert_to_tensor=True)
        snips = [e["snippet"] for e in evidence]
        snip_emb = self.model.encode(snips, convert_to_tensor=True)

        scores = util.cos_sim(claim_emb, snip_emb)[0]

        for i, e in enumerate(evidence):
            e["score"] = float(scores[i])

        ranked = sorted(evidence, key=lambda x: x["score"], reverse=True)

        return ranked[:5]


# =========================
# SIMPLE VERIFIER (NO LLM DEPENDENCY)
# =========================
class ClaimVerifier:

    def verify(self, claim, evidence):

        if not evidence:
            return "UNVERIFIED", 0.2, "No evidence found."

        support = []
        contradict = []

        claim_lower = claim.lower()

        for e in evidence:

            text = e["snippet"].lower()
            domain = e["domain"]

            # -------------------------
            # STRONG CONTRADICTION
            # -------------------------
            if any(w in text for w in [
                # research words
               "study", "research", "found",
              "linked", "associated", "evidence",

              # health effect words (IMPORTANT FIX)
             "increase risk", "cause", "leads to",
             "heart", "anxiety", "side effects",
             "danger", "harm", "negative effects",
                "impact", "affect"
            ]):
                support.append(domain)
                continue

            # -------------------------
            # 🔥 CRITICAL FIX: EXTREME CLAIM
            # -------------------------
            if "10 cups" in claim_lower:

                # if evidence talks about normal range → contradiction
                if any(w in text for w in [
                    "2 cups", "3 cups", "4 cups",
                    "moderate", "recommended",
                    "limit", "safe amount"
                ]):
                    contradict.append(domain)
                    continue

                # if evidence does NOT mention 10 cups → also contradiction
                if "10 cups" not in text:
                    contradict.append(domain)
                    continue

            # -------------------------
            # SUPPORT (only if strong match)
            # -------------------------
            if any(w in text for w in [
                "study", "research", "found",
                "linked", "associated", "evidence"
            ]):
                support.append(domain)

        # remove duplicates
        support = list(set(support))
        contradict = list(set(contradict))

        # -------------------------
        # FINAL DECISION
        # -------------------------
        if len(contradict) > len(support):
            verdict = "FAKE"
        else:
            verdict = "REAL"

        # -------------------------
        # CONFIDENCE
        # -------------------------
        confidence = 0.5 + 0.1 * len(support) - 0.1 * len(contradict)
        confidence = max(0.0, min(confidence, 1.0))

        # -------------------------
        # EXPLANATION
        # -------------------------
        explanation = f"Based on retrieved sources, the claim is {verdict}. "
        explanation += f"{len(support)} sources support and {len(contradict)} contradict."

        if "10 cups" in claim_lower:
            explanation += " The claim is exaggerated and not supported by scientific evidence."

        explanation += f"\nSupporting sources: {', '.join(support) if support else 'None'}"
        explanation += f"\nContradicting sources: {', '.join(contradict) if contradict else 'None'}"

        return verdict, confidence, explanation


# =========================
# OUTPUT
# =========================
class OutputGenerator:

    def display(self, claim, verdict, confidence, explanation, evidence):

        print("\n==============================")
        print("FAKE NEWS DETECTION RESULT")
        print("==============================")

        print("\nClaim:", claim)
        print("\nVerdict:", verdict)
        print("Confidence:", round(confidence,2))

        print("\nExplanation:")
        print(explanation)

        print("\n--- SOURCE ATTRIBUTION ---")
        for i,e in enumerate(evidence,1):
            print(f"{i}. {e['domain']}")
            print(f"   {e['url']}")


# =========================
# PIPELINE
# =========================
class FakeNewsPipeline:

    def __init__(self):
        self.extractor = ClaimExtractor()
        self.query = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()
        self.output = OutputGenerator()

    def detect(self, article):

        claims = self.extractor.extract_claims(article)

        final_results = []

        for claim in claims:

            queries = self.query.generate_queries([claim])

            evidence = []
            for q in queries:
                evidence.extend(self.retriever.search(q))

            ranked = self.ranker.rank(claim, evidence)

            verdict, conf, expl = self.verifier.verify(claim, ranked)

            self.output.display(claim, verdict, conf, expl, ranked)

            final_results.append(verdict)

        final = "FAKE" if final_results.count("FAKE") > final_results.count("REAL") else "REAL"

        print("\n==============================")
        print("FINAL VERDICT:", final)
        print("==============================")

In [33]:
pipeline = FakeNewsPipeline()

article = """
Scientists report that moderate coffee consumption (3–4 cups daily) may improve longevity.
However, some viral posts claim drinking 10 cups of coffee daily can extend lifespan by 50 years.
Health experts warn that excessive caffeine intake can cause heart problems and anxiety.
"""

result = pipeline.detect(article)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



FAKE NEWS DETECTION RESULT

Claim: Scientists report that moderate coffee consumption (3–4 cups daily) may improve longevity

Verdict: REAL
Confidence: 0.8

Explanation:
Based on retrieved sources, the claim is REAL. 3 sources support and 0 contradict.
Supporting sources: Prevention, Economic Times, MedicalNewsToday
Contradicting sources: None

--- SOURCE ATTRIBUTION ---
1. MedicalNewsToday
   https://www.medicalnewstoday.com/articles/roundup-coffee-and-longevity-3-studies-explore-how-coffee-may-benefit-healthy-aging
2. Prevention
   https://www.prevention.com/health/a69729904/three-to-four-cups-of-coffee-slows-biological-aging-study/
3. Economic Times
   https://economictimes.indiatimes.com/news/international/us/drinking-34-cups-of-coffee-a-day-may-help-you-live-longer-study-finds/articleshow/125989913.cms
4. Economic Times
   https://economictimes.indiatimes.com/news/international/us/drinking-34-cups-of-coffee-a-day-may-help-you-live-longer-study-finds/articleshow/125989913.cms
5. F